# MWE — Resource estimation (`qarp.resources`)

One call: circuit → **stage-explicit resource vectors** (qubits, depth, gate/2q/T/SWAP counts).

Three pieces:

- **`ResourceVector`** — the frozen, versioned contract (`to_dict()` is the wire format
  external tools consume; pinned by a golden test).
- **`ResourceEstimator` / `estimate()`** — runs the *real* compilation pipeline
  (rebase → optimize → route → rebase) and snapshots a counted vector at every stage
  boundary. The numbers are ground truth of what would execute, not a parallel cost model.
- **`ResourceModeler`** — engine-specific cost models fill the *modeled* fields
  (e.g. a digital-Rz T-cost). None ships with qarp; an engine may advertise one via
  `engine.resource_modeler()`, and `estimate(modeler=…)` takes any conforming object.

Two semantics worth internalizing: **`None` is not `0`** (`swap_count=None` means
"routing hasn't happened"), and **counted vs modeled never mix** (`t_count` comes from
gates in the circuit; `t_count_modeled` from a modeler, tagged in provenance).

In [ ]:
import math

import qarpx as qx
from qarp.blocks import SimpleBlock
from qarp.devices import Device
from qarp.devices import get_nearest_neighbour_architecture
from qarp.resources import Stage, estimate


def fanout_block(n=5):
    """CX(0, k) fan-out + two Rz — forces SWAPs on a line architecture."""
    b = SimpleBlock(n)
    b.h(0)
    for k in range(1, n):
        b.cx(0, k)
    b.rz(3, 1e-3)          # two Rz of very different size — a cost
    b.rz(4, 0.3)           # model may price them differently
    b.measure([(q, q) for q in range(n)])
    return b


block = fanout_block()
report = estimate(block)   # no gateset/device: LOGICAL snapshot only
report[Stage.LOGICAL]

## Full pipeline on a device

Give the estimator a gate set and a device and it stages the actual pipeline.
`swap_count` exists **only at ROUTED** — the final rebase decomposes each router
SWAP into 3 CX, so at TARGET the overhead lives in `n_2q` instead. `t_count` is
`None` throughout: an `Rz` remains in the circuit, so a counted T number would be a lie.

In [ ]:
device = Device(5, architecture=get_nearest_neighbour_architecture(1, 5))
report = estimate(
    block,
    gateset=qx.clifford_t_rz_gateset(),
    device=device,
    device_label="line-5",
)

header = f"{'stage':<10} {'width':>5} {'depth':>5} {'gates':>5} {'2q':>4} {'swap':>5} {'T':>5}"
print(header + "\n" + "-" * len(header))
for stage, v in report.items():
    fmt = lambda x: "—" if x is None else x
    print(f"{stage.value:<10} {v.n_qubits:>5} {v.depth:>5} {v.n_gates:>5} "
          f"{v.n_2q:>4} {fmt(v.swap_count):>5} {fmt(v.t_count):>5}")

## Modeled fields: the `ResourceModeler` extension point

`t_count_modeled` and `extras` are never *counted* — they come from a **modeler**: an object
with a `name` and a `model(commands, vector)` method that returns the vector with those fields
set and `provenance.modeler` naming it. No modeler ships with qarp; `estimate(modeler=…)` runs
whatever you pass on the *final* stage only (§19: counted and modeled never mix). The class
below is the whole contract — note that a `Custom` gate must **null** the count rather than
undercount, since rotations fused inside it are invisible to any per-gate pricing.


In [ ]:
class OneTPerRz:
    """Toy modeler: one T per Rz.  A real one prices a synthesis budget."""

    name = "one_t_per_rz"

    def model(self, commands, vector):
        n_rz = sum(c.gate == qx.GateType.Rz for c in commands)
        opaque = any(c.gate == qx.GateType.Custom for c in commands)
        return vector.with_model(
            modeler=self.name,
            t_count_modeled=None if opaque else float(n_rz),
            extras={"n_rz": float(n_rz)},
        )


report = estimate(
    block,
    gateset=qx.clifford_t_rz_gateset(),
    device=device,
    device_label="line-5",
    modeler=OneTPerRz(),
)
final = report.final
print("stage:          ", final.provenance.stage.value)
print("modeler:        ", final.provenance.modeler)
print("t_count:        ", final.t_count, " (counted — None: Rz gates remain)")
print("t_count_modeled:", final.t_count_modeled)
print("extras:         ", final.extras)


## Exact Clifford+T synthesis: the SYNTHESIZED stage

Pass `synthesis_epsilon` (with the Clifford+T+Rz gate set) and the estimator appends a real
Ross–Selinger synthesis stage: every remaining `Rz` is replaced by an H/S/T/X
sequence within ε of the exact rotation (per rotation, operator norm, global phase
included). No `Rz` survives, so `t_count` becomes an **exact integer** — counted,
not modeled. Needs the optional `pygridsynth` dependency
(`pip install "openqarp[cliffordt]"`).

In [ ]:
report = estimate(
    block,
    gateset=qx.clifford_t_rz_gateset(),
    device=device,
    device_label="line-5",
    synthesis_epsilon=1e-10,
)
syn = report[Stage.SYNTHESIZED]
print("synthesis:      ", syn.provenance.synthesis)
print("t_count:        ", syn.t_count, " (exact — no Rz left)")
print("histogram:      ", dict(sorted(syn.op_histogram.items())))
# ~3·log2(1/eps) T's per non-Clifford rotation (Ross & Selinger 2016):
print("expected/rot:   ", round(3 * math.log2(1e10)))

## The wire format

`report.to_dict()` / `ResourceVector.to_dict()` is the versioned schema external
tools consume — flat scalars, a raw `op_histogram`
detail channel, and a self-describing `provenance` block.

In [ ]:
import json

print(json.dumps(final.to_dict(), indent=2))

**Schema stability**: the dict layout is pinned by a golden test
(`tests/test_resources/test_vector.py`) under `schema_version = 1`. Changes are
additive-with-version-bump, never silent.